<a href="https://www.kaggle.com/code/ameythakur20/tpu-flower-classification-advanced-ensemble" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Petals to the Metal: TPU Flower Classification with Dual-Backbone Ensembling

**EfficientNet and DenseNet with CutMixUp, class-balanced focal loss, pseudo-label fine-tuning, and test-time augmentation**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

This notebook studies a dual-backbone ensemble for flower classification on TPU under the macro F1 evaluation used in the competition. The central objective is to improve minority-class performance while preserving a training procedure that remains practical within a single Kaggle TPU session.

The current pipeline combines backbone-specific preprocessing, CutMix and MixUp regularization, class-balanced focal loss, macro F1-based checkpoint selection, pseudo-label fine-tuning on the unlabeled test set, and averaged test-time augmentation at inference. The notebook is written as an applied research notebook, with each engineering choice motivated by its expected effect on optimization stability or class-wise balance.

**Outline:**

1. [Execution Environment Initialization](#1.-Execution-Environment-Initialization)
2. [Hardware Synchronization and Distribution Strategy](#2.-Hardware-Synchronization-and-Distribution-Strategy)
3. [Global Hyperparameter Configuration](#3.-Global-Hyperparameter-Configuration)
4. [Statistical Overview of Dataset Distribution](#4.-Statistical-Overview-of-Dataset-Distribution)
5. [Stochastic Regularization and Data Ingestion](#5.-Stochastic-Regularization-and-Data-Ingestion)
6. [Visualization of Augmented Tensors](#6.-Visualization-of-Augmented-Tensors)
7. [Cyclical Optimization Dynamics](#7.-Cyclical-Optimization-Dynamics)
8. [Dual-Stream Architectural Assembly](#8.-Dual-Stream-Architectural-Assembly)
9. [Model Convergence and Evaluation](#9.-Model-Convergence-and-Evaluation)
10. [Inference via Test Time Augmentation](#10.-Inference-via-Test-Time-Augmentation)
11. [Summary](#11.-Summary)

## 1. Execution Environment Initialization

To establish proper hardware assignment for TensorFlow distributed strategies, framework-level hardware locks generated by external compilation libraries (e.g., JAX) must be explicitly bypassed via system environment state configuration. This ensures the TPU Matrix Multiplication Units (MMUs) remain accessible exclusively for TensorFlow operation kernels.

In [ ]:
# Install the TPU runtime bridge expected by the Kaggle TensorFlow image.
# TensorFlow is kept as the sole accelerator client so that TPU initialization remains predictable.
!export PATH="${HOME}/.local/bin:${PATH}" && \
    uv pip install --system tensorflow-tpu==2.18.0 --find-links https://storage.googleapis.com/libtpu-tf-releases/index.html && \
    uv pip install --system "ml_dtypes>=0.5.1"

import os
import logging
import warnings

# JAX is pinned to CPU because competing accelerator clients can interfere with TPU reservation.
os.environ['JAX_PLATFORMS'] = 'cpu'
# The legacy Keras flag matches the runtime expected by the Kaggle TensorFlow image.
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print('TPU runtime packages installed and accelerator ownership configured for TensorFlow.')

## 2. Hardware Synchronization and Distribution Strategy

The numerical stack is initialized explicitly before model construction so that TensorFlow can reserve and manage the TPU runtime consistently. Mixed precision is enabled because TPUs execute bfloat16 matrix operations efficiently, while the final classifier output is later kept in float32 to preserve stable probability estimates.

In [ ]:
import tensorflow as tf
import numpy as np
import math
import re
import matplotlib.pyplot as plt
from kaggle_datasets import KaggleDatasets

# No explicit mixed-precision policy is set here.
# On Kaggle TPU, XLA already lowers many matrix operations efficiently, while float32 model construction avoids dtype mismatches in application backbones.
tf.get_logger().setLevel(logging.ERROR)

plt.style.use('fivethirtyeight')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'figure.facecolor': '#ffffff',
    'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#f0f0f0',
    'grid.color': '#e0e0e0',
    'text.color': '#2c3e50',
    'axes.labelcolor': '#2c3e50',
    'xtick.color': '#7f8c8d',
    'ytick.color': '#7f8c8d',
    'legend.framealpha': 1.0
})

try:
    # The local TPU resolver is the standard entry point on Kaggle TPU VMs.
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)

    print(f'Status: Successfully initialized. Running on TPU: {tpu.master()}')
    print(f'Synchronous Execution Replicas: {strategy.num_replicas_in_sync}')
except Exception as e:
    print(f'Hardware Initialization Status: Execution error. {e}')
    raise RuntimeError('Execution halted because a TPU runtime was not detected.')

## 3. Global Hyperparameter Configuration

The training configuration is organized around TPU throughput, macro F1 sensitivity, and the memory cost of a dual-backbone model. Resolution, batch size, focal-loss parameters, pseudo-label thresholds, and inference-time averaging factors are declared here so that the full training schedule can be modified from a single location.

In [ ]:
IMAGE_SIZE = [512, 512]
EPOCHS = 8
PSEUDO_EPOCHS = 3
SEED = 42
AUTO = tf.data.AUTOTUNE
CLASSES = 104

# A moderate per replica batch size leaves memory headroom for the dual backbone and batch-level regularization.
BASE_BATCH_SIZE = 4
BATCH_SIZE = BASE_BATCH_SIZE * strategy.num_replicas_in_sync
# A larger steps_per_execution value reduces Python overhead during TPU training.
STEPS_PER_EXECUTION = 16

# These parameters control label mixing, label smoothing, focal reweighting, and inference blending.
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
CUTMIX_PROB = 0.5
LABEL_SMOOTHING = 0.05
FOCAL_GAMMA = 2.0
PSEUDO_LABEL_THRESHOLD = 0.97
TTA_STEPS = 5
PHASE1_BLEND_WEIGHT = 0.45
PHASE2_BLEND_WEIGHT = 0.55

# The maximum learning rate is scaled by replica count to preserve the effective update size on TPU.
LR_START = 1e-5
LR_MAX = 5e-5 * strategy.num_replicas_in_sync
LR_MIN = 1e-6
PSEUDO_LR_MAX = 2e-5 * strategy.num_replicas_in_sync

tf.random.set_seed(SEED)
np.random.seed(SEED)

def get_robust_dataset_path():
    # Multiple mount points are checked because Kaggle datasets may be attached under different canonical paths.
    paths = [
        f'/kaggle/input/tpu-getting-started/tfrecords-jpeg-{IMAGE_SIZE[0]}x{IMAGE_SIZE[0]}',
        f'/kaggle/input/competitions/tpu-getting-started/tfrecords-jpeg-{IMAGE_SIZE[0]}x{IMAGE_SIZE[0]}',
        f'/kaggle/input/flower-classification-with-tpus/tfrecords-jpeg-{IMAGE_SIZE[0]}x{IMAGE_SIZE[0]}'
    ]

    try:
        gcs_path = KaggleDatasets().get_gcs_path('tpu-getting-started')
        paths.append(gcs_path + f'/tfrecords-jpeg-{IMAGE_SIZE[0]}x{IMAGE_SIZE[0]}')
    except Exception:
        pass

    for path in paths:
        if tf.io.gfile.exists(path):
            return path
    return paths[0]

DATASET_PATH = get_robust_dataset_path()

def find_tfrecords(path, split):
    # Both subdirectory and prefix layouts appear in public Kaggle copies of this competition dataset.
    patterns = [
        f'{path}/{split}/*.tfrec',
        f'{path}/{split}*.tfrec'
    ]
    files = []
    for pattern in patterns:
        files += tf.io.gfile.glob(pattern)
    return sorted(files)

TRAINING_FILENAMES = find_tfrecords(DATASET_PATH, 'train')
VALIDATION_FILENAMES = find_tfrecords(DATASET_PATH, 'val')
TEST_FILENAMES = find_tfrecords(DATASET_PATH, 'test')
ALL_LABELED_FILENAMES = sorted(TRAINING_FILENAMES + VALIDATION_FILENAMES)

print(f'Active Dataset Path: {DATASET_PATH}')
print(f'File Counts: Train({len(TRAINING_FILENAMES)}) Val({len(VALIDATION_FILENAMES)}) Test({len(TEST_FILENAMES)})')

## 4. Statistical Overview of Dataset Distribution

The number of records in each split is computed directly from the TFRecord shard names so that epoch boundaries can be defined without scanning the full dataset. A simple inverse-frequency class prior is also estimated from the training split, since macro F1 is sensitive to errors on classes that appear only rarely.

In [ ]:
def count_data_items(filenames):
    # Sample counts are parsed from the shard names so epoch boundaries can be computed without reading the full dataset.
    n = [int(re.compile(r'-([0-9]*)\.').search(filename).group(1)) for filename in filenames]
    return np.sum(n)

n_train = count_data_items(TRAINING_FILENAMES)
n_val = count_data_items(VALIDATION_FILENAMES)
n_test = count_data_items(TEST_FILENAMES)
n_labeled = n_train + n_val

print(f'Training samples: {n_train} | Validation samples: {n_val} | Test samples: {n_test}')

def plot_dataset_distribution(train, val, test):
    fig, ax = plt.subplots(figsize=(10, 6))
    segments = ['Training', 'Validation', 'Testing']
    counts = [train, val, test]
    colors = ['#2980b9', '#27ae60', '#e67e22']

    bars = ax.bar(segments, counts, color=colors, width=0.55)

    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval + (yval * 0.02), int(yval),
                ha='center', va='bottom', fontweight='bold', fontsize=12, color='#2c3e50')

    ax.set_title('Observations per Dataset Split', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Total Image Observations', fontsize=13)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_dataset_distribution(n_train, n_val, n_test)

def estimate_class_weights(filenames):
    # Minority classes contribute equally under macro F1, so an inverse-frequency prior is computed from the training split.
    dataset = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.batch(256)

    class_totals = np.zeros(CLASSES, dtype=np.float32)
    for _, batch_labels in dataset:
        class_totals += np.sum(batch_labels.numpy(), axis=0)

    weights = np.sum(class_totals) / (CLASSES * np.maximum(class_totals, 1.0))
    weights = weights / np.mean(weights)
    return tf.constant(weights, dtype=tf.float32)

## 5. Stochastic Regularization and Data Ingestion

The input pipeline is designed to keep TPU execution saturated while introducing label-preserving variation in both image content and target distributions. Mild photometric augmentation is applied at the example level, while MixUp and CutMix are applied after batching so that the model is exposed to smoother decision boundaries and less overconfident predictions.

In [ ]:
def decode_image(image_data):
    # Images are decoded to float32 in [0, 1] so that augmentation remains numerically stable and backend preprocessing can be applied later.
    image = tf.image.decode_jpeg(image_data, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

def read_labeled_tfrecord(example):
    labeled_tfrecord_format = {
        'image': tf.io.FixedLenFeature([], tf.string),
        'class': tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, labeled_tfrecord_format)
    image = decode_image(example['image'])
    # One-hot targets are required because MixUp, CutMix, and focal loss operate on soft label distributions.
    label = tf.one_hot(tf.cast(example['class'], tf.int32), CLASSES)
    return image, tf.cast(label, tf.float32)

def read_unlabeled_tfrecord(example):
    unlabeled_tfrecord_format = {
        'image': tf.io.FixedLenFeature([], tf.string),
        'id': tf.io.FixedLenFeature([], tf.string),
    }
    example = tf.io.parse_single_example(example, unlabeled_tfrecord_format)
    image = decode_image(example['image'])
    idnum = example['id']
    return image, idnum

def augment_image(image):
    # The augmentation budget stays lightweight so the input pipeline remains faster than the training step on TPU.
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.20)
    image = tf.image.random_contrast(image, lower=0.80, upper=1.20)
    image = tf.image.random_saturation(image, lower=0.75, upper=1.25)
    image = tf.image.random_hue(image, max_delta=0.03)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image

def sample_beta_distribution(size, concentration):
    # MixUp and CutMix draw interpolation coefficients from a Beta distribution, implemented here via two Gamma samples.
    gamma_1 = tf.random.gamma(shape=[size], alpha=concentration)
    gamma_2 = tf.random.gamma(shape=[size], alpha=concentration)
    return gamma_1 / (gamma_1 + gamma_2)

def mixup_batch(images, labels, alpha=MIXUP_ALPHA):
    batch_size = tf.shape(images)[0]
    mixing_weights = sample_beta_distribution(batch_size, alpha)
    image_weights = tf.reshape(mixing_weights, [batch_size, 1, 1, 1])
    label_weights = tf.reshape(mixing_weights, [batch_size, 1])
    shuffled_indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = images * image_weights + tf.gather(images, shuffled_indices) * (1.0 - image_weights)
    mixed_labels = labels * label_weights + tf.gather(labels, shuffled_indices) * (1.0 - label_weights)
    return mixed_images, mixed_labels
def cutmix_batch(images, labels, alpha=CUTMIX_ALPHA):
    shuffled_indices = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    shuffled_images = tf.gather(images, shuffled_indices)
    shuffled_labels = tf.gather(labels, shuffled_indices)

    lambda_value = sample_beta_distribution(1, alpha)[0]
    cut_ratio = tf.math.sqrt(1.0 - lambda_value)
    cut_width = tf.cast(IMAGE_SIZE[1] * cut_ratio, tf.int32)
    cut_height = tf.cast(IMAGE_SIZE[0] * cut_ratio, tf.int32)

    center_x = tf.random.uniform([], 0, IMAGE_SIZE[1], dtype=tf.int32)
    center_y = tf.random.uniform([], 0, IMAGE_SIZE[0], dtype=tf.int32)

    x1 = tf.clip_by_value(center_x - cut_width // 2, 0, IMAGE_SIZE[1])
    x2 = tf.clip_by_value(center_x + cut_width // 2, 0, IMAGE_SIZE[1])
    y1 = tf.clip_by_value(center_y - cut_height // 2, 0, IMAGE_SIZE[0])
    y2 = tf.clip_by_value(center_y + cut_height // 2, 0, IMAGE_SIZE[0])

    # A single patch geometry is applied to the full batch, which keeps the operation vectorized and inexpensive.
    mask = tf.pad(
        tf.ones([y2 - y1, x2 - x1, 3], dtype=images.dtype),
        [[y1, IMAGE_SIZE[0] - y2], [x1, IMAGE_SIZE[1] - x2], [0, 0]]
    )
    mask = tf.expand_dims(mask, axis=0)

    mixed_images = images * (1.0 - mask) + shuffled_images * mask
    lambda_adjusted = 1.0 - tf.cast((x2 - x1) * (y2 - y1), tf.float32) / tf.cast(IMAGE_SIZE[0] * IMAGE_SIZE[1], tf.float32)
    mixed_labels = labels * lambda_adjusted + shuffled_labels * (1.0 - lambda_adjusted)
    return mixed_images, mixed_labels

def apply_batch_regularization(images, labels):
    # A stochastic switch between CutMix and MixUp exposes the classifier to two complementary forms of label interpolation.
    selector = tf.random.uniform([], 0.0, 1.0)
    return tf.cond(selector < CUTMIX_PROB,
                   lambda: cutmix_batch(images, labels),
                   lambda: mixup_batch(images, labels))

def with_dataset_options(dataset, deterministic):
    options = tf.data.Options()
    options.experimental_deterministic = deterministic
    return dataset.with_options(options)

def build_labeled_dataset(filenames, training=False):
    dataset = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTO)
    dataset = with_dataset_options(dataset, deterministic=not training)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    if training:
        dataset = dataset.map(lambda image, label: (augment_image(image), label), num_parallel_calls=AUTO)
    return dataset

def build_pseudo_labeled_dataset(probabilities, threshold):
    confidences = tf.constant(np.max(probabilities, axis=1).astype(np.float32))
    pseudo_labels = tf.constant(np.eye(CLASSES, dtype=np.float32)[np.argmax(probabilities, axis=1)])

    dataset = tf.data.TFRecordDataset(TEST_FILENAMES, num_parallel_reads=AUTO)
    dataset = with_dataset_options(dataset, deterministic=True)
    dataset = dataset.map(read_unlabeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.map(lambda image, idnum: image, num_parallel_calls=AUTO)
    dataset = dataset.enumerate()
    dataset = dataset.filter(lambda index, image: tf.greater_equal(tf.gather(confidences, index), threshold))
    dataset = dataset.map(
        lambda index, image: (augment_image(image), tf.gather(pseudo_labels, index)),
        num_parallel_calls=AUTO
    )
    return dataset

def get_training_dataset(filenames, pseudo_probabilities=None, pseudo_threshold=None):
    dataset = build_labeled_dataset(filenames, training=True)
    if pseudo_probabilities is not None and pseudo_threshold is not None:
        dataset = dataset.concatenate(build_pseudo_labeled_dataset(pseudo_probabilities, pseudo_threshold))
    dataset = dataset.shuffle(4096, seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.repeat()
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
    dataset = dataset.map(apply_batch_regularization, num_parallel_calls=AUTO)
    dataset = dataset.prefetch(AUTO)
    return dataset

def get_validation_dataset(filenames=VALIDATION_FILENAMES):
    dataset = build_labeled_dataset(filenames, training=False)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

def get_test_dataset(ordered=False):
    dataset = tf.data.TFRecordDataset(TEST_FILENAMES, num_parallel_reads=AUTO)
    dataset = with_dataset_options(dataset, deterministic=ordered)
    dataset = dataset.map(read_unlabeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

## 6. Visualization of Augmented Tensors

A small visualization batch is assembled after preprocessing so that the geometric and photometric transformations can be inspected before training. This step is especially useful when label mixing is introduced, because it provides an immediate qualitative check that the pipeline is modifying the data in the intended range.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

BOTANICAL_CLASSES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea', 'wild geranium', 'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood', 'globe thistle',
    'snapdragon', "colt's foot", 'king protea', 'spear thistle', 'yellow iris', 'globe-flower', 'purple coneflower', 'peruvian lily', 'balloon flower', 'giant white arum lily',
    'fire lily', 'pincushion flower', 'fritillary', 'red ginger', 'grape hyacinth', 'corn poppy', 'prince of wales feathers', 'stemless gentian', 'artichoke', 'sweet william',
    'carnation', 'garden phlox', 'love in the mist', 'cosmos', 'alpine sea holly', 'ruby-lipped cattleya', 'cape flower', 'great masterwort', 'siam tulip', 'lenten rose',
    'barberton daisy', 'daffodil', 'sword lily', 'poinsettia', 'bolero deep blue', 'wallflower', 'marigold', 'buttercup', 'daisy', 'common dandelion',
    'petunia', 'wild pansy', 'primula', 'sunflower', 'lilac hibiscus', 'bishop of llandaff', 'gaura', 'geranium', 'orange dahlia', 'pink-yellow dahlia',
    'cautleya spicata', 'japanese anemone', 'black-eyed susan', 'silverbush', 'californian poppy', 'osteospermum', 'spring crocus', 'iris', 'windflower', 'tree poppy',
    'gazania', 'azalea', 'water lily', 'rose', 'thorn apple', 'morning glory', 'passion flower', 'lotus', 'toad lily', 'anthurium',
    'frangipani', 'clematis', 'hibiscus', 'columbine', 'desert-rose', 'tree mallow', 'magnolia', 'cyclamen', 'watercress', 'canna lily',
    'hippeastrum', 'bee balm', 'pink quill', 'foxglove', 'bougainvillea', 'camellia', 'mallow', 'mexican petunia', 'bromelia', 'blanket flower',
    'trumpet creeper', 'blackberry lily', 'common tulip', 'wild rose'
]

CLASSES = len(BOTANICAL_CLASSES)

def batch_to_numpy_images_and_labels(data):
    images, labels = data
    numpy_images = images.numpy()
    numpy_labels = labels.numpy()
    if numpy_labels.dtype == object:
        numpy_labels = [None for _ in enumerate(numpy_images)]
    return numpy_images, numpy_labels

def title_from_label_and_target(predicted_label, target_label):
    if target_label is None:
        return BOTANICAL_CLASSES[predicted_label], True
    predicted_index = int(predicted_label)
    target_index = int(np.argmax(target_label)) if np.ndim(target_label) > 0 else int(target_label)
    is_correct = predicted_index == target_index
    return f'{BOTANICAL_CLASSES[predicted_index]} [' + ('OK' if is_correct else 'NO') + f'] {BOTANICAL_CLASSES[target_index]}', is_correct

def display_one_flower(image, title, subplot, red=False, titlesize=16):
    plt.subplot(*subplot)
    plt.axis('off')
    plt.imshow(image)
    if title:
        plt.title(title, fontsize=titlesize, color='red' if red else 'black', fontdict={'verticalalignment': 'center'}, pad=titlesize / 1.5)
    return (subplot[0], subplot[1], subplot[2] + 1)

def display_batch_of_images(databatch, predictions=None):
    images, labels = batch_to_numpy_images_and_labels(databatch)
    if labels is None:
        labels = [None for _ in enumerate(images)]

    rows = int(math.sqrt(len(images)))
    cols = len(images) // rows

    figsize = 13.0
    spacing = 0.1
    subplot = (rows, cols, 1)
    if rows < cols:
        plt.figure(figsize=(figsize, figsize / cols * rows))
    else:
        plt.figure(figsize=(figsize / rows * cols, figsize))

    for index, (image, label) in enumerate(zip(images[:rows * cols], labels[:rows * cols])):
        title = '' if label is None else BOTANICAL_CLASSES[int(np.argmax(label))]
        is_correct = True
        if predictions is not None:
            title, is_correct = title_from_label_and_target(predictions[index], label)

        dynamic_titlesize = figsize * spacing / max(rows, cols) * 40 + 3
        subplot = display_one_flower(image, title, subplot, not is_correct, titlesize=dynamic_titlesize)

    plt.tight_layout()
    plt.subplots_adjust(wspace=spacing, hspace=spacing)
    plt.show()

def display_training_batch():
    try:
        training_dataset_visualization = get_training_dataset(TRAINING_FILENAMES).unbatch().batch(16)
        train_batch = next(iter(training_dataset_visualization))
        display_batch_of_images(train_batch)
    except StopIteration:
        print('Status: Execution error. Training dataset is empty. Verify dataset attachment and permissions.')
    except Exception as e:
        print(f'Visualization Status: Execution error. {e}')

display_training_batch()

## 7. Cyclical Optimization Dynamics

Learning-rate scheduling is used to separate the unstable early phase of optimization from the later fine-tuning phase. A short warmup reduces abrupt updates at the start of training, and exponential decay then decreases the step size gradually as the ensemble approaches a more stable solution.

In [ ]:
LR_RAMPUP_EPOCHS = 4
LR_SUSTAIN_EPOCHS = 0
LR_EXP_DECAY = 0.80

def lrfn(epoch):
    # The warmup stage avoids unstable early updates, after which exponential decay supports gradual fine-tuning.
    if epoch < LR_RAMPUP_EPOCHS:
        lr = (LR_MAX - LR_START) / LR_RAMPUP_EPOCHS * epoch + LR_START
    elif epoch < LR_RAMPUP_EPOCHS + LR_SUSTAIN_EPOCHS:
        lr = LR_MAX
    else:
        lr = (LR_MAX - LR_MIN) * LR_EXP_DECAY ** (epoch - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS) + LR_MIN
    return lr

lr_callback = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=True)

epochs_range = np.arange(EPOCHS)
lrs = [lrfn(epoch) for epoch in epochs_range]

plt.figure(figsize=(10, 5))
plt.plot(epochs_range, lrs, marker='o', markersize=6, color='#8e44ad', linewidth=2.5, label='Learning Rate')
plt.fill_between(epochs_range, lrs, color='#8e44ad', alpha=0.15)
plt.title('Learning Rate Schedule', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.7)
plt.xticks(np.arange(0, EPOCHS + 1, max(1, EPOCHS // 4)))
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 8. Dual-Stream Architectural Assembly

The classifier is formed by combining two pretrained convolutional backbones with different representational biases. EfficientNet contributes scale-efficient feature extraction, while DenseNet contributes densely reused features; their pooled representations are concatenated and passed through a compact classification head trained under a class-balanced focal objective.

In [ ]:
from tensorflow.keras.applications import EfficientNetB6, DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Concatenate, Dropout, BatchNormalization, Lambda
from tensorflow.keras.models import Model

CLASS_WEIGHT_VECTOR = estimate_class_weights(TRAINING_FILENAMES)

# Focal loss is used because macro F1 is sensitive to minority-class errors that standard cross-entropy tends to underweight.
def categorical_focal_loss(y_true, y_pred, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING):
    y_true = tf.cast(y_true, tf.float32)
    if label_smoothing > 0:
        y_true = y_true * (1.0 - label_smoothing) + label_smoothing / tf.cast(CLASSES, tf.float32)

    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
    alpha = tf.cast(CLASS_WEIGHT_VECTOR, tf.float32)
    cross_entropy = -y_true * tf.math.log(y_pred) * alpha
    focal_weight = tf.pow(1.0 - y_pred, gamma)
    return tf.reduce_sum(focal_weight * cross_entropy, axis=-1)

def build_model():
    with strategy.scope():
        input_tensor = Input(shape=[*IMAGE_SIZE, 3])

        # The shared input tensor follows the global mixed-precision policy, so each backbone branch is cast back to float32 before application preprocessing.
        float32_input = Lambda(lambda x: tf.cast(x, tf.float32), name='float32_input_cast')(input_tensor)

        # Each backbone then receives the value range expected by its pretrained ImageNet weights.
        efficientnet_input = Lambda(lambda x: x * 255.0, dtype='float32', name='efficientnet_input_scale')(float32_input)
        densenet_input = Lambda(lambda x: densenet_preprocess(x * 255.0), dtype='float32', name='densenet_input_preprocess')(float32_input)

        base_model_1 = EfficientNetB6(weights='imagenet', include_top=False, input_tensor=efficientnet_input)
        base_model_1.trainable = True
        pool_1 = GlobalAveragePooling2D()(base_model_1.output)

        base_model_2 = DenseNet201(weights='imagenet', include_top=False, input_tensor=densenet_input)
        base_model_2.trainable = True
        pool_2 = GlobalAveragePooling2D()(base_model_2.output)

        # The two backbones are intentionally different so that the merged representation captures less correlated errors.
        merged = Concatenate()([pool_1, pool_2])
        merged = BatchNormalization()(merged)
        merged = Dropout(0.50)(merged)
        merged = Dense(512, activation='relu')(merged)
        merged = Dropout(0.30)(merged)
        # The classifier output is kept in float32 so probability computations remain stable under mixed precision.
        output = Dense(CLASSES, activation='softmax', dtype='float32')(merged)

        model = Model(inputs=input_tensor, outputs=output)
        optimizer = tf.keras.optimizers.Adam(learning_rate=LR_START)

        model.compile(
            optimizer=optimizer,
            loss=categorical_focal_loss,
            metrics=[
                tf.keras.metrics.CategoricalAccuracy(name='categorical_accuracy'),
                tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')
            ],
            steps_per_execution=STEPS_PER_EXECUTION
        )
    return model

model = build_model()

## 9. Model Convergence and Evaluation

Model selection follows the competition metric rather than relying only on loss or accuracy. Validation macro F1 is computed explicitly at the end of each epoch, and the corresponding weights are restored before the pseudo-label stage so that the second training phase starts from the strongest validation checkpoint.

In [ ]:
STEPS_PER_EPOCH = max(1, n_train // BATCH_SIZE)
VALIDATION_STEPS = max(1, math.ceil(n_val / BATCH_SIZE))

print('Starting distributed training session...')
print(f'Training Config: {STEPS_PER_EPOCH} steps/epoch | {EPOCHS} epochs | batch size {BATCH_SIZE}')
print(f'Validation Config: {VALIDATION_STEPS} steps/epoch')

def macro_f1_score(y_true, y_pred, num_classes=CLASSES):
    # Macro F1 is computed explicitly so model selection follows the competition metric rather than accuracy alone.
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(confusion, (y_true, y_pred), 1)

    true_positives = np.diag(confusion)
    false_positives = confusion.sum(axis=0) - true_positives
    false_negatives = confusion.sum(axis=1) - true_positives

    precision = np.divide(true_positives, true_positives + false_positives, out=np.zeros(num_classes, dtype=np.float64), where=(true_positives + false_positives) != 0)
    recall = np.divide(true_positives, true_positives + false_negatives, out=np.zeros(num_classes, dtype=np.float64), where=(true_positives + false_negatives) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(num_classes, dtype=np.float64), where=(precision + recall) != 0)
    return float(np.mean(f1))

class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, validation_dataset, patience=3):
        super().__init__()
        self.validation_dataset = validation_dataset
        self.patience = patience
        self.best_score = -np.inf
        self.best_weights = None
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        true_labels = []
        pred_labels = []
        for batch_images, batch_labels in self.validation_dataset:
            batch_predictions = self.model(batch_images, training=False).numpy()
            true_labels.append(np.argmax(batch_labels.numpy(), axis=1))
            pred_labels.append(np.argmax(batch_predictions, axis=1))

        true_labels = np.concatenate(true_labels)
        pred_labels = np.concatenate(pred_labels)
        score = macro_f1_score(true_labels, pred_labels)

        logs = logs or {}
        logs['val_macro_f1'] = score
        print(f'val_macro_f1: {score:.5f}')

        if score > self.best_score:
            self.best_score = score
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print('Early stopping on validation macro F1.')
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            # Restoring the strongest macro F1 checkpoint keeps the pseudo-label stage aligned with the competition metric.
            self.model.set_weights(self.best_weights)
            print(f'Restored best model weights with val_macro_f1={self.best_score:.5f}')

def pseudo_lrfn(epoch):
    if PSEUDO_EPOCHS == 1:
        return PSEUDO_LR_MAX
    decay = epoch / max(1, PSEUDO_EPOCHS - 1)
    return PSEUDO_LR_MAX * (0.5 ** decay)

pseudo_lr_callback = tf.keras.callbacks.LearningRateScheduler(pseudo_lrfn, verbose=True)

validation_dataset = get_validation_dataset(VALIDATION_FILENAMES)
macro_f1_callback = MacroF1Callback(validation_dataset, patience=3)

history = model.fit(
    get_training_dataset(TRAINING_FILENAMES),
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    callbacks=[lr_callback, macro_f1_callback],
    validation_data=validation_dataset,
    validation_steps=VALIDATION_STEPS,
    verbose=1
)

## 10. Inference via Test Time Augmentation

Inference is carried out by averaging predictions across several mild, label-preserving transformations of each test image. The notebook also performs a second inference pass after pseudo-label fine-tuning and blends the two probability tensors, which reduces sensitivity to any single training phase.

In [ ]:
def apply_tta(image, tta_index):
    # Each TTA branch applies a mild, label-preserving perturbation so that prediction averaging reduces variance without distorting morphology.
    if tta_index % 2 == 1:
        image = tf.image.flip_left_right(image)
    if tta_index % 3 == 1:
        image = tf.image.adjust_brightness(image, 0.04)
    if tta_index % 3 == 2:
        image = tf.image.adjust_contrast(image, 1.08)
    return tf.clip_by_value(image, 0.0, 1.0)

def get_test_dataset_tta(tta_index, ordered=True):
    dataset = tf.data.TFRecordDataset(TEST_FILENAMES, num_parallel_reads=AUTO)
    dataset = with_dataset_options(dataset, deterministic=ordered)
    dataset = dataset.map(read_unlabeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.map(lambda image, idnum: (apply_tta(image, tta_index), idnum), num_parallel_calls=AUTO)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

def predict_with_tta(model_to_score):
    probabilities = np.zeros((n_test, CLASSES), dtype=np.float32)
    for tta_index in range(TTA_STEPS):
        print(f'TTA iteration: {tta_index + 1}/{TTA_STEPS}')
        tta_ds = get_test_dataset_tta(tta_index, ordered=True).map(lambda image, idnum: image, num_parallel_calls=AUTO)
        probabilities += model_to_score.predict(tta_ds, verbose=0) / TTA_STEPS
    return probabilities

phase1_probabilities = predict_with_tta(model)
phase1_confidence = np.max(phase1_probabilities, axis=1)
pseudo_count = int(np.sum(phase1_confidence >= PSEUDO_LABEL_THRESHOLD))
print(f'Pseudo labels selected: {pseudo_count} / {n_test} at threshold {PSEUDO_LABEL_THRESHOLD:.2f}')

if pseudo_count > 0:
    pseudo_training_steps = max(1, math.ceil((n_labeled + pseudo_count) / BATCH_SIZE))
    model.fit(
        get_training_dataset(ALL_LABELED_FILENAMES, pseudo_probabilities=phase1_probabilities, pseudo_threshold=PSEUDO_LABEL_THRESHOLD),
        steps_per_epoch=pseudo_training_steps,
        epochs=PSEUDO_EPOCHS,
        callbacks=[pseudo_lr_callback],
        verbose=1
    )

phase2_probabilities = predict_with_tta(model)
final_probabilities = (PHASE1_BLEND_WEIGHT * phase1_probabilities) + (PHASE2_BLEND_WEIGHT * phase2_probabilities)
predictions = np.argmax(final_probabilities, axis=-1)

test_ds = get_test_dataset(ordered=True)
test_ids_ds = test_ds.map(lambda image, idnum: idnum, num_parallel_calls=AUTO).unbatch()
test_ids = next(iter(test_ids_ds.batch(n_test))).numpy().astype('U')

output_structure = np.rec.fromarrays([test_ids, predictions])
np.savetxt('submission.csv', output_structure, fmt=['%s', '%d'], delimiter=',', header='id,label', comments='')
print('Submission file generated from averaged TTA probabilities.')

## 11. Summary

1. **Distributed Computation Strategy**: Implemented TPU distribution mapping for synchronized training across replicas.
2. **Dual-Backbone Representation**: Combined EfficientNet and DenseNet features after applying the preprocessing expected by each pretrained network.
3. **Regularization Strategy**: Added color jitter, MixUp, CutMix, and class-balanced focal weighting to improve robustness on minority classes.
4. **Model Selection**: Selected the training checkpoint with the strongest validation macro F1.
5. **Inference Procedure**: Performed pseudo-label fine-tuning, averaged TTA predictions, and generated the final submission.csv file.

---

**Citation:**
Alexis Cook, Phil Culliton, and Ryan Holbrook. Petals to the Metal - Flower Classification on TPU.
https://kaggle.com/competitions/tpu-getting-started, 2020. Kaggle.